# ReTone v3 — DDSP timbre-decoder training (48 kHz) · Engine 1 (mono-pitched family)

Train a **per-instrument timbre decoder** that re-voices a stem as that instrument while **preserving the original performance** (pitch curve, dynamics, vibrato). One pipeline, every instrument **in this family**.

### Scope — which stems this engine is for
A DDSP *harmonic+noise* decoder is single-voiced by construction (one f0 → one harmonic series). That makes it the **highest-quality** tool for **sustained / monophonic-pitched** sound and the wrong tool for chords or percussion. So Engine 1 covers:

- ✅ **violin, flute, saxophone, trumpet, bass**, and re-voicing **monophonic stems** (vocals, bass, mono lead) as any of them.
- ❌ **piano, guitar** (polyphonic — hammer/pluck attacks; stacking mono voices sounds "stacked") → **Engine 2: AFTER** (control-transfer diffusion, spectral/latent-domain).
- ❌ **drums** (percussion — no f0 to drive DDSP) → **Engine 3: Inverse Drum Machine** (onset+velocity analysis → one-shot swap).

All three engines share one philosophy — *analyze the input, resynthesize with timbre decoupled from performance* — but they are different models per family (verified: DDSP is a dead end for struck/plucked strings and percussion). This notebook builds Engine 1; Engines 2–3 are integration (pretrained checkpoints), not training.

### What you produce
`<instrument>48.pth` (plain `state_dict`) + `<instrument>48.yaml` → drop into the worker's `runpod/ddsp_models/` + a 1-line `MODELS` edit. Nothing else in the worker changes.

### Where to run
A **RunPod GPU Pod, 24 GB (RTX 4090 / A5000)**, PyTorch base image. Start by cloning your repo on the pod so this notebook and the ported components travel together:
```bash
git clone https://github.com/Ganeshveer/retone.git
cd retone/notebooks     # this notebook is here
```
> Each instrument is a separate multi-hour run. Recommended order: **train violin first as a quality gate**, A/B it against the current v2, then batch the rest.


## 0 · Configuration

Set the instrument + capacity here. Everything downstream reads `CFG`. To train a different instrument later, change `INSTRUMENT` and re-run from §2.

In [ ]:
import os, sys, subprocess, pathlib, json, shutil, textwrap

# ── pick the instrument for THIS run ────────────────────────────────────────
INSTRUMENT = "violin"     # violin | flute | saxophone | trumpet | bass | piano | guitar | <your own>

# ── paths (relative to the pod's working dir) ───────────────────────────────
ROOT      = pathlib.Path("/workspace/retone_train").resolve()   # scratch root on the pod
REPO_DIR  = pathlib.Path("..").resolve()                        # the retone repo (this nb is in retone/notebooks)
SC_DIR    = ROOT / "ddsp-pytorch"                               # sweetcocoa clone
DATA_DIR  = ROOT / "data" / INSTRUMENT
CKPT_DIR  = ROOT / "ckpt" / f"{INSTRUMENT}48"
CFG_DIR   = ROOT / "configs_48k"
OUT_DIR   = ROOT / "artifacts"                                  # final .pth + .yaml land here
for d in (ROOT, DATA_DIR/"raw", DATA_DIR/"train", DATA_DIR/"test", CKPT_DIR, CFG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── 48 kHz max-quality capacity (see recipe §4) ─────────────────────────────
CFG = dict(
    sample_rate=48000,        # native 48k — recovers the 8–24 kHz band 16k throws away
    frame_resolution=0.004,   # KEEP: hop=int(sr*fr)=192, integer 4ms CREPE step, 48000%192==0
    n_harmonics=180,          # up from 101 → richer high end (main VRAM lever)
    n_freq=128,               # up from 65 → finer noise filter-bank for the wider band
    gru_units=512, mlp_layers=3, mlp_units=512,
    use_z=False,              # KEEP False: avoids pitch leakage + the hardcoded MFCC f_max=8000 bug
    use_reverb=True,          # worker calls reconstruction(add_reverb=True) and reads audio_reverb
    n_fft=2048, n_mels=128, n_mfcc=30, z_units=16,   # z-encoder-only (inert while use_z=False)
    crepe="full", bidirectional=False,
    # training
    waveform_sec=1.0, batch_size=32, num_step=200000,
    # 32 (up from the recipe default 16) measured live: ~1.4-2x samples/sec on a pod
    # with GPU/VRAM headroom (was 48% util, 4.4/20.5GB). num_step is a fixed STEP count,
    # not a data budget, so this does not make training hit step 200000 sooner — it means
    # each step sees more audio. Net effect: check quality proportionally EARLIER (e.g.
    # ~50-75k steps, not the recipe's original 100-150k) rather than waiting for a higher
    # absolute step count. If VRAM/CPU headroom differs on your pod, re-measure sustained
    # it/s over a FULL validation_interval before trusting any batch_size/num_workers change
    # — short post-restart snapshots are misleadingly fast and do not hold up.
    valid_waveform_sec=6, validation_interval=1000,
    loss="mss", optimizer="adam",   # adam dodges radam's deprecated in-place ops on modern torch
    lr=0.001, lr_decay=0.98, lr_min=1.0e-07, lr_scheduler="multi",
    metric="mss", f0_threshold=0.5, num_workers=16, resume=True, seed=940513,
    # 16 workers measured faster than 6 (which matches this pod's real ~7.65-core cgroup
    # quota) despite nominally oversubscribing it — used the empirically faster value.
)
print("INSTRUMENT =", INSTRUMENT)
print("ROOT       =", ROOT)
print("repo       =", REPO_DIR, "(exists:", (REPO_DIR/"runpod"/"ddsp_engine.py").exists(), ")")


## 1 · Environment

A clean, low-friction stack (**torch 2.1.2**) so the sweetcocoa code needs only tiny patches. The trained file is a plain `state_dict`, so this torch version need **not** match the worker's 2.7.1.

In [ ]:
# GPU + torch. (RunPod PyTorch pods usually ship torch already; we pin a known-good pair.)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
!pip -q install torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
# f0 CLI (CREPE is TensorFlow-backed — training box only; the worker never gets TF).
# NOTE: "tensorflow==2.15.*" alone frequently fails to pull matching CUDA/cuDNN wheels via pip
# and silently falls back to CPU (no error — just ~10-50x slower, easy to mistake for "stuck").
# The [and-cuda] extra is TF's own documented way to get the GPU build via pip on Linux.
!pip -q install "crepe" "tensorflow[and-cuda]==2.15.*"
# ddsp-pytorch deps + audio tooling
!pip -q install omegaconf easydict tensorboardX numpy pandas tqdm soundfile librosa soxr scipy pyloudnorm requests
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# IMPORTANT: check TF's GPU visibility in an ISOLATED subprocess, not an inline `import tensorflow`
# in this cell. Torch is already imported above — if TF is ALSO imported into this same process,
# their bundled CUDA/cuDNN/cuBLAS libs can conflict ("Unable to register cuDNN factory... already
# registered") and TF falls back to CPU, even though the real `crepe` CLI (invoked later via a
# fresh subprocess.run, never sharing a process with torch) would see the GPU just fine. Testing
# in a fresh process here mirrors what `crepe` actually experiences.
import subprocess
r = subprocess.run(["python3", "-c",
                     "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"],
                    capture_output=True, text=True)
print("TF GPUs visible (isolated process):", r.stdout.strip() or r.stderr[-300:])


In [ ]:
# Clone sweetcocoa/ddsp-pytorch and apply the patches (recipe §2 + fixes found live on a
# real pod run — this 2020-era repo predates modern torch/omegaconf/pandas by several
# major versions; every patch below was hit and confirmed via an actual training run).
import subprocess, shutil, pathlib, re
if not SC_DIR.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/sweetcocoa/ddsp-pytorch.git",str(SC_DIR)], check=True)

comp_src = REPO_DIR/"runpod"/"vendor"/"ddsp_pytorch"/"components"
comp_dst = SC_DIR/"components"
assert comp_src.exists(), f"ported components not found at {comp_src} — are you running from retone/notebooks?"

# PATCH 1 — reuse OUR torch.fft-ported components so train-time math == worker inference math.
for fn in ("reverb.py","loudness_extractor.py","filtered_noise.py"):
    shutil.copy2(comp_src/fn, comp_dst/fn)
    print("copied ported", fn)

# PATCH 2 — torchaudio 2.x renamed load(offset=…) → load(frame_offset=…)
ad = SC_DIR/"train"/"dataset"/"audiodata.py"
s = ad.read_text()
s2 = re.sub(r"torchaudio\.load\(([^)]*?)\boffset\s*=", r"torchaudio.load(\1frame_offset=", s)
if s2 != s: ad.write_text(s2); print("patched audiodata.py: offset= → frame_offset=")
else: print("audiodata.py: no offset= found (already ok)")

def _patch_exact(path, replacements, label):
    s = path.read_text()
    for old, new in replacements:
        if old not in s:
            print(f"  {label}: pattern not found (already patched or upstream changed) — {old[:60]!r}")
            continue
        s = s.replace(old, new)
    path.write_text(s)
    print(f"patched {label}")

# PATCH 3 — OmegaConf 2.x removed the instance .save() method (io.py calls it with real
# expressions, e.g. `logpath[:-4] + ".yaml"`, not literal strings — a naive regex misses these).
_patch_exact(SC_DIR/"train"/"trainer"/"io.py", [
    ('conf.save(logpath[:-4] + ".yaml")', 'OmegaConf.save(conf, logpath[:-4] + ".yaml")'),
    ('conf.save(args.ckpt + ".yaml")',    'OmegaConf.save(conf, args.ckpt + ".yaml")'),
], "io.py (conf.save -> OmegaConf.save)")

# PATCH 4 — OmegaConf 2.x removed DictConfig.pretty() (replaced by OmegaConf.to_yaml()).
_patch_exact(SC_DIR/"train"/"train.py", [
    ('OmegaConf.create(config.__dict__).pretty()', 'OmegaConf.to_yaml(OmegaConf.create(config.__dict__))'),
], "train.py (.pretty() -> OmegaConf.to_yaml())")

# PATCH 5 — sub-modules (e.g. LoudnessExtractor) bake a hardcoded self.device="cpu" default
# that Encoder never overrides when constructing them, so their internal constant tensors
# (A-weighting curve) stay on CPU while the real input audio is on GPU -> device-mismatch
# crash mid-training. Same fix already applied in the worker's ddsp_engine.py; train.py
# needs the equivalent realignment right after the model is moved to GPU.
_patch_exact(SC_DIR/"train"/"train.py", [
    (
        "net = AutoEncoder(config).cuda()",
        'net = AutoEncoder(config).cuda()\n'
        '# sub-modules bake a hardcoded self.device="cpu" default Encoder never overrides —\n'
        '# align every submodule to the real device (mirrors ddsp_engine.py\'s inference fix).\n'
        '_net_device = next(net.parameters()).device\n'
        'for _m in net.modules():\n'
        '    if hasattr(_m, "device"):\n'
        '        _m.device = _net_device'
    ),
], "train.py (device realignment)")

# PATCH 6 — pandas >=2.0 removed DataFrame.append() (used twice: per-step logging, and
# the cross-experiment CSV merge in update_experiment()). Both replaced with pd.concat().
_patch_exact(SC_DIR/"train"/"trainer"/"trainer.py", [
    (
        "self.dataframe = self.dataframe.append(\n"
        "                dict(zip(kwarg_list, print_kwarg)), ignore_index=True\n"
        "            )",
        "self.dataframe = pd.concat(\n"
        "                [self.dataframe, pd.DataFrame([dict(zip(kwarg_list, print_kwarg))])],\n"
        "                ignore_index=True,\n"
        "            )",
    ),
    ("df_ex = df_ex.append(df_config, sort=False)", "df_ex = pd.concat([df_ex, df_config], sort=False)"),
], "trainer.py (DataFrame.append -> pd.concat)")

# PATCH 7 — enable cudnn.benchmark (free autotune speedup for our fixed input shapes —
# waveform_sec never changes within a run).
#
# Also tuned live on a real run: batch_size 16->32 + num_workers 4->16. `nproc` reports
# 48 cores, but the container's REAL cgroup CPU quota (/sys/fs/cgroup/cpu.max) was only
# ~7.65 — nproc reflects the HOST's total, not this container's allocation. Measured
# sustained samples/sec (it/s x batch_size, over 60-90s windows — short post-restart
# snapshots are misleadingly fast and don't hold up) across three configs:
#   batch=16 workers=4  (original): ~41-43 samples/sec
#   batch=32 workers=16          :  ~81   samples/sec  <- fastest, kept despite
#                                                          oversubscribing the real quota
#   batch=32 workers=6 (matches quota): ~59 samples/sec
# num_step is a fixed step COUNT, not a data budget, so this does not make the run hit
# step 200000 sooner — each step now just sees ~2x the audio. The real win: check
# quality proportionally EARLIER (recipe's "good by 100-150k" becomes ~50-75k here).
_patch_exact(SC_DIR/"train"/"train.py", [
    (
        "net = AutoEncoder(config).cuda()",
        "torch.backends.cudnn.benchmark = True\n"
        "net = AutoEncoder(config).cuda()",
    ),
], "train.py (cudnn.benchmark)")

print("\nall patches done.")


## 2 · Data

**Target:** ~15 min of clean, consistent, single-performer **solo** phrases per instrument, at 48 kHz mono. *Consistency* (one instrument / player / room / mic) matters more than quantity; >20 min gives diminishing returns.

Two ways to fill `data/<instrument>/raw/` — use either or both:

1. **Drop your own** (primary, works for any instrument incl. **bass**): copy cleaned solo WAV/FLAC/MP3 files into the `raw/` folder printed below. For **monophonic** instruments give solo playing; for **piano/guitar** prefer *single-note / melodic* passages so the timbre decoder learns cleanly (polyphony is handled later at inference).
2. **Auto-fetch URMP** (optional, native 48 kHz, for violin/flute/trumpet/saxophone): the helper cell pulls URMP's isolated solo stems via the Dryad API and keeps only this instrument's `AuSep_*` files.

> This is a research project — grab the highest-quality audio you can regardless of license.


In [ ]:
print("Put raw solo recordings for", INSTRUMENT, "here:\n ", DATA_DIR/"raw")
print("Accepted: .wav .flac .mp3 .aiff — any sample rate/channels (we resample to 48k mono next).")
existing = sorted([p.name for p in (DATA_DIR/"raw").glob("*") if p.suffix.lower() in (".wav",".flac",".mp3",".aiff",".aif")])
print(f"\ncurrently {len(existing)} file(s) in raw/:", existing[:12], "…" if len(existing)>12 else "")


### 2.0 · Fetching the full URMP archive once (shared across all 4 instruments)

`Dataset.tar.gz` is **12.1 GB** (one archive holding violin + flute + trumpet + saxophone + everything else) — download it **once** here, and every future instrument run (§0 → change `INSTRUMENT` → re-run) reuses the same extracted copy instead of re-fetching.

**Two things worth knowing before you run this:**

1. **It won't cost you any personal data / home bandwidth**, and it's cheap on RunPod specifically because **RunPod charges $0 for both ingress and egress bandwidth** on pods — you only pay for the GPU-seconds and disk you already have running, not for moving bytes in or out. The download happens entirely between Dryad's servers and RunPod's datacenter; your laptop's connection is never in the path.
2. **A plain `curl`/`requests` GET will NOT work** — verified directly against Dryad's live API: the versioned API (`/api/v2/files/{id}/download`) requires an OAuth bearer token (`401 Unauthorized, must have current bearer token`), and the public website's download route (`/downloads/file_stream/{id}`) is now sitting behind an **Anubis anti-bot JS challenge** (you'll get back an HTML "Checking..." page, not the tarball, if you hit it blind). Both are real, current findings — not a guess — so the cell below picks one of two working routes rather than a naive `wget`:

- **Route A — reuse a browser's cookie (fastest to set up, recommended):** open `https://datadryad.org/dataset/doi:10.5061/dryad.ng3r749` in a normal browser once, let the "Checking your browser…" page clear (a few seconds — this is the ONLY part that touches your home connection, and it's just a small HTML/JS page, not the 12 GB file), then in DevTools → Network tab, find the request to `file_stream/99348` (or just reload the page) and copy its **`Cookie` request header** value. Paste it into `DRYAD_COOKIE` below. The pod then does the entire 12 GB pull using that cookie.
- **Route B — Dryad API OAuth token (fully scriptable, no browser at all):** create a free account at [datadryad.org](https://datadryad.org), then under your profile create an "API application" to get a `client_id`/`client_secret`, and paste them into `DRYAD_CLIENT_ID`/`DRYAD_CLIENT_SECRET` below. This is documented (`POST /oauth/token`, `grant_type=client_credentials`) and — being the official API path rather than the public web route — may not hit the Anubis gate at all. I could not personally verify this end-to-end (it needs your own Dryad credentials to test), so the cell validates the download before trusting it either way.

Either route, the cell below **verifies the result is really a gzip archive of the right size** before extracting — if Dryad serves back an HTML challenge/error page instead, it fails loudly with a clear message rather than silently corrupting your data directory.

In [ ]:
# Fetch Dataset.tar.gz ONCE, validated, then extract for every instrument to share.
import requests, tqdm

URMP_DIR = ROOT/"urmp_full"; URMP_DIR.mkdir(exist_ok=True)
TAR_PATH = URMP_DIR/"Dataset.tar.gz"
EXTRACT_DIR = URMP_DIR/"extracted"
DRYAD_FILE_ID = 99348                 # verified via the Dryad API: path="Dataset.tar.gz"
DRYAD_EXPECTED_SIZE = 12097672571     # verified via the Dryad API "size" field (~11.3 GiB)

# --- fill in ONE of these two routes (see markdown above) ---
DRYAD_COOKIE = ""            # Route A: paste the Cookie header from a solved browser session
DRYAD_CLIENT_ID = ""         # Route B: from your Dryad profile -> API applications
DRYAD_CLIENT_SECRET = ""

UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36"

def _get_oauth_token():
    r = requests.post("https://datadryad.org/oauth/token", data={
        "grant_type": "client_credentials",
        "client_id": DRYAD_CLIENT_ID,
        "client_secret": DRYAD_CLIENT_SECRET,
    }, timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]

def _download(headers):
    # Route B redirects through a signed S3 URL (dryad-assetstore-merritt-*.s3.amazonaws.com);
    # `requests` follows redirects by default and correctly drops the Authorization header when
    # the host changes, so no special handling is needed here — just follow and stream.
    url = f"https://datadryad.org/api/v2/files/{DRYAD_FILE_ID}/download" if "Authorization" in headers \
          else f"https://datadryad.org/downloads/file_stream/{DRYAD_FILE_ID}"
    part = TAR_PATH.with_suffix(".tar.gz.part")
    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        # NOTE: do NOT gate on the Content-Type header here — verified live that Dryad's S3
        # backend serves this file with Content-Type: text/plain even though the bytes are a
        # genuine gzip stream (confirmed via a byte-level probe: starts with \x1f\x8b). The
        # header is just mislabeled at the source, so the only trustworthy check is the actual
        # bytes, done below after download — content-type is logged for visibility only.
        print(f"  response: {r.status_code} {r.headers.get('content-type','?')!r} "
              f"content-length={r.headers.get('content-length','?')}")
        total = int(r.headers.get("content-length", 0))
        with open(part, "wb") as f, tqdm.tqdm(total=total, unit="B", unit_scale=True, desc="Dataset.tar.gz") as bar:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk); bar.update(len(chunk))
    part.rename(TAR_PATH)

def fetch_urmp_full():
    if TAR_PATH.exists() and TAR_PATH.stat().st_size == DRYAD_EXPECTED_SIZE:
        print("already have a valid Dataset.tar.gz — skipping download.")
        return

    if DRYAD_CLIENT_ID and DRYAD_CLIENT_SECRET:
        print("using Route B (OAuth token)...")
        token = _get_oauth_token()
        _download({"Authorization": f"Bearer {token}"})
    elif DRYAD_COOKIE:
        print("using Route A (browser cookie)...")
        _download({"User-Agent": UA, "Cookie": DRYAD_COOKIE})
    else:
        raise RuntimeError(
            "Set DRYAD_COOKIE (Route A) or DRYAD_CLIENT_ID/DRYAD_CLIENT_SECRET (Route B) above — "
            "see the markdown cell right above this one for exactly how to get either."
        )

    # The ONLY real validation: actual magic bytes + exact size. Protects against silently
    # extracting an HTML error/challenge page as if it were the real archive — this is what
    # would catch a blocked/interstitial response, unlike the (unreliable) Content-Type header.
    size = TAR_PATH.stat().st_size
    with open(TAR_PATH, "rb") as f:
        magic = f.read(2)
    if magic != b"\x1f\x8b" or size != DRYAD_EXPECTED_SIZE:
        bad = TAR_PATH.read_bytes()[:300]
        TAR_PATH.unlink()
        raise RuntimeError(
            f"download did not validate (magic={magic!r}, size={size} vs expected {DRYAD_EXPECTED_SIZE}). "
            f"Deleted the bad file. First 300 bytes were:\n{bad}"
        )
    print(f"✓ verified Dataset.tar.gz — {size/1e9:.2f} GB, valid gzip.")

fetch_urmp_full()


In [ ]:
# Extract the FULL archive once (shared by every instrument — no re-download needed).
import tarfile

if EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
    print(f"already extracted at {EXTRACT_DIR} — skipping.")
else:
    EXTRACT_DIR.mkdir(exist_ok=True)
    print("extracting (12+ GB, several minutes)...")
    with tarfile.open(TAR_PATH) as t:
        t.extractall(EXTRACT_DIR)
    print("done:", sum(1 for _ in EXTRACT_DIR.rglob("*.wav")), "wav files extracted")


In [ ]:
# Populate THIS instrument's raw/ from the shared extraction (symlinks — no copying, no re-download).
# URMP tags: vn=violin, fl=flute, tpt=trumpet, sax=saxophone. Re-run this cell after changing
# INSTRUMENT in §0 to pull that instrument's stems from the already-downloaded archive.
URMP_TAG = {"violin": "vn", "flute": "fl", "trumpet": "tpt", "saxophone": "sax"}.get(INSTRUMENT)

if URMP_TAG is None:
    print(f"no URMP tag for '{INSTRUMENT}' — skip this cell and use your own recordings (see §2 above).")
else:
    matches = sorted(EXTRACT_DIR.rglob(f"AuSep_*_{URMP_TAG}_*.wav"))
    linked = 0
    for src in matches:
        dst = DATA_DIR/"raw"/src.name
        if not dst.exists():
            dst.symlink_to(src)
            linked += 1
    print(f"{INSTRUMENT}: {len(matches)} URMP stems available, {linked} newly linked into {DATA_DIR/'raw'}")
    if len(matches) * 20 < 8 * 60:   # crude: URMP clips avg ~20s; want >=8 min = 480s total
        print("⚠️  URMP coverage for this instrument may be thin (<8 min) — consider adding your own recordings too.")


### 2.1 · Preprocess → 48 kHz mono, trim, normalize, split

Order matters — DDSP is sensitive to input hygiene. Silence has undefined f0 and poisons the loudness stats, so we trim it.

In [ ]:
import soundfile as sf, librosa, numpy as np, glob, random, subprocess, pyloudnorm as pyln
random.seed(CFG["seed"])

raw_files = [p for p in (DATA_DIR/"raw").glob("*") if p.suffix.lower() in (".wav",".flac",".mp3",".aiff",".aif")]
assert raw_files, f"no audio in {DATA_DIR/'raw'} — add files (see §2) or enable URMP fetch."

def prep_one(src, dst):
    # 48k mono via ffmpeg (high-quality soxr resampler)
    tmp = dst.with_suffix(".tmp.wav")
    subprocess.run(["ffmpeg","-y","-loglevel","fatal","-i",str(src),
                    "-ac","1","-ar","48000","-af","aresample=resampler=soxr","-sample_fmt","s16",str(tmp)], check=True)
    y, sr = sf.read(tmp); os.remove(tmp)
    if y.ndim>1: y = y.mean(1)
    # trim leading/trailing + long internal silence
    y,_ = librosa.effects.trim(y, top_db=35)
    if len(y) < sr*1.2:   # skip clips shorter than a training crop
        return 0.0
    # peak-normalize to ~ -1 dBFS
    peak = np.max(np.abs(y)) + 1e-9
    y = (y/peak*0.89).astype(np.float32)
    sf.write(dst, y, 48000, subtype="PCM_16")
    return len(y)/48000.0

random.shuffle(raw_files)
n_test = max(1, int(round(len(raw_files)*0.1)))
splits = {"test": raw_files[:n_test], "train": raw_files[n_test:]}
mins = {"train":0.0, "test":0.0}
for split, files in splits.items():
    out = DATA_DIR/split
    for f in out.glob("*.wav"): f.unlink()
    for i,src in enumerate(files):
        dur = prep_one(src, out/f"{split}_{i:04d}.wav")
        mins[split]+=dur
print(f"train: {mins['train']/60:.1f} min ({len(splits['train'])} files) | test: {mins['test']/60:.1f} min ({len(splits['test'])} files)")
if mins['train']/60 < 8:
    print("\n⚠️  <8 min of training audio — quality will suffer. Add more consistent solo recordings.")


### 2.2 · CREPE f0 → `f0_0.004/` CSVs

The **only** cached feature (loudness is computed inside the model). CREPE internally resamples to 16 kHz and returns f0 on a 4 ms time grid, so the CSV is identical for 16 k and 48 k. The folder name **must** be `f0_0.004` to match `frame_resolution`.

In [ ]:
# CREPE CLI writes <name>.f0.csv (time,frequency,confidence) per wav, at --step-size ms.
# Streams output live (no capture_output) and prints file counts up front — a silent multi-file
# subprocess otherwise looks identical whether it's working or stuck. If TF can't see a GPU
# (checked in §1), this step can take ~10-50x longer per file — check nvidia-smi if it feels slow.
for split in ("train","test"):
    d = DATA_DIR/split
    f0d = d/"f0_0.004"; f0d.mkdir(exist_ok=True)
    n_wav = len(list(d.glob("*.wav")))
    print(f"CREPE f0 on {split}/ — {n_wav} files, model-capacity={CFG['crepe']!r}, viterbi …")
    r = subprocess.run(["crepe", str(d), "--output", str(f0d), "--viterbi", "--step-size", "4", "--model-capacity", CFG["crepe"]])
    csvs = list(f0d.glob("*.f0.csv"))
    print(f"  → {len(csvs)}/{n_wav} f0 csv(s) written")
    assert csvs, "CREPE produced no CSVs — check the crepe/tensorflow install above."


## 3 · Write the 48 kHz config

This exact file is **both** the training config **and** the yaml you ship to the worker. They must be byte-identical on the capacity keys — `load_state_dict(strict=False)` silently random-inits any layer whose shape disagrees, giving garbage with no error.

In [ ]:
from omegaconf import OmegaConf
cfg = dict(CFG)
cfg.update(
    experiment_name=f"DDSP_{INSTRUMENT}_48k",
    train=str(DATA_DIR/"train")+"/", test=str(DATA_DIR/"test")+"/",
    ckpt=str(CKPT_DIR/"v1.pth"), tensorboard_dir=str(ROOT/"tensorboard_log")+"/",
    gpu=0,  # verified: rented pods here are single-GPU (index 0) — --gpu 1 sets
            # CUDA_VISIBLE_DEVICES=1, which doesn't exist -> "No CUDA GPUs available"
    seed=CFG["seed"],
)
cfg_path = CFG_DIR/f"{INSTRUMENT}48.yaml"
OmegaConf.save(OmegaConf.create(cfg), cfg_path)
print("wrote", cfg_path)
print(OmegaConf.to_yaml(OmegaConf.create(cfg)))


## 4 · Fast sanity check (do this FIRST — ~10 min)

Before spending GPU-hours, prove the whole 48 k pipeline (FFT ports, config shapes, CREPE grid, hop=192) end-to-end by **overfitting a single file** for 2 000 steps. The MSS loss should drop steeply and the logged reconstruction should already resemble the input. If this breaks, the full run would too.

In [ ]:
# Overfit one clean file for 2000 steps. Uses a tiny temp data dir with a single train/test wav.
import shutil
sanity = ROOT/"sanity"/INSTRUMENT
for sp in ("train","test"):
    (sanity/sp/"f0_0.004").mkdir(parents=True, exist_ok=True)
one = sorted((DATA_DIR/"train").glob("*.wav"))[0]
one_csv = (DATA_DIR/"train"/"f0_0.004"/(one.stem+".f0.csv"))
for sp in ("train","test"):
    for f in (sanity/sp).glob("*.wav"): f.unlink()
    for f in (sanity/sp/"f0_0.004").glob("*.csv"): f.unlink()
    shutil.copy2(one, sanity/sp/one.name)
    shutil.copy2(one_csv, sanity/sp/"f0_0.004"/one_csv.name)

sane = dict(cfg); sane.update(train=str(sanity/"train")+"/", test=str(sanity/"test")+"/",
                              ckpt=str(ROOT/"ckpt"/"sanity.pth"), num_step=2000, batch_size=4,
                              validation_interval=500, valid_waveform_sec=2, experiment_name=f"SANITY_{INSTRUMENT}")
sane_path = CFG_DIR/f"{INSTRUMENT}_sanity.yaml"
OmegaConf.save(OmegaConf.create(sane), sane_path)   # kept as a reference artifact + used by §4.1 below

def cfg_to_cli(d):
    """train.py has NO --config flag: setup() auto-generates one argparse flag per key
    from a default config HARDCODED into the source (always ../configs/violin.yaml
    inside the clone, regardless of which instrument we're training — that file only
    supplies the flag SCHEMA/defaults; every value is overridden explicitly here).
    Boolean keys are auto-typed as store_true (if that file's default is False) or
    store_false (if its default is True) — so presence of the flag ALWAYS INVERTS the
    baked default, it never simply "sets True". Verified live via `train.py --help` +
    reading trainer/io.py's get_args(). `resume` is additionally special: passing it
    makes train.py unconditionally `trainer.load(ckpt)` with NO existence check, so it
    must only be present when a checkpoint genuinely already exists on disk."""
    baked = OmegaConf.load(SC_DIR/"configs"/"violin.yaml")
    BOOL_KEYS = {"bidirectional", "use_reverb", "use_z", "resume"}
    d = dict(d)
    d["resume"] = os.path.exists(d["ckpt"])  # only resume if there's actually something to resume from
    args = []
    for k, v in d.items():
        if k in BOOL_KEYS:
            if bool(v) != bool(baked.get(k, False)):
                args.append(f"--{k}")
        else:
            args += [f"--{k}", str(v)]
    return args

import sys
cmd = [sys.executable, "train.py"] + cfg_to_cli(sane)
print("running:", " ".join(cmd), "\n(cwd = %s/train)\n" % SC_DIR)
r = subprocess.run(cmd, cwd=str(SC_DIR/"train"))
print("\nsanity exit code:", r.returncode, "— 0 = pipeline is sound, launch the full run in §5.")


### 4.1 · Round-trip the sanity model through the worker engine

Confirms the checkpoint loads into our `AutoEncoder(cfg)` with no silent `strict=False` garbage, and that `render_mono` produces 48 k→44.1 k audio that tracks the input pitch (timbre will be rough at 2 k steps — that's fine, we're checking the *plumbing*).

In [ ]:
# Load the 2k-step checkpoint through the REAL worker engine and render a test file.
import numpy as np, soundfile as sf
sys.path.insert(0, str(REPO_DIR/"runpod"))
# point the engine at our sanity artifacts
import importlib, ddsp_engine as de
de.MODELS = {INSTRUMENT: ("sanity.pth", f"{INSTRUMENT}_sanity.yaml")}
de._MODELS_DIR = str(ROOT/"ckpt")            # where sanity.pth lives
shutil.copy2(sane_path, ROOT/"ckpt"/f"{INSTRUMENT}_sanity.yaml")
eng = de.DDSPEngine()
test_wav = sorted((DATA_DIR/"test").glob("*.wav"))[0]
y, sr = sf.read(test_wav)
out, osr = eng.render_mono(np.asarray(y, dtype=np.float32), sr, INSTRUMENT)
sf.write(ROOT/"sanity_render.wav", out, osr)
print(f"rendered {len(out)/osr:.1f}s @ {osr} Hz | RMS={np.sqrt(np.mean(out**2)):.4f} peak={np.max(np.abs(out)):.3f}")
print("→ non-silent audio that tracks the input pitch = plumbing verified. Listen:", ROOT/"sanity_render.wav")
import IPython.display as ipd; ipd.Audio(str(ROOT/"sanity_render.wav"))


## 5 · Full training run

Now the real thing — **200 k steps** at max-quality capacity. Expect **~5–8 h on a 4090**; quality is often already good by 100–150 k. Watch the validation **MSS loss** in TensorBoard and **early-stop when it plateaus for ~20 k steps** — and trust your ears on the logged validation audio, which is the real metric. `resume: true` picks up from the checkpoint if the pod restarts.

In [ ]:
# TensorBoard (open the pod's forwarded port, or run this in a separate cell/terminal)
%load_ext tensorboard
%tensorboard --logdir {str(ROOT/"tensorboard_log")} --port 6006


In [ ]:
# Full 200k-step run. Safe to re-run/resume — cfg_to_cli() only passes --resume once
# CKPT_DIR/v1.pth actually exists on disk (see cfg_to_cli's definition in §4 above).
cmd = [sys.executable, "train.py"] + cfg_to_cli(cfg)
print("running:", " ".join(cmd), "\n")
# If you hit CUDA OOM: lower batch_size in CFG (§0) to 8, re-run §3 to rewrite the yaml, then retry here.
r = subprocess.run(cmd, cwd=str(SC_DIR/"train"))
print("\ntrain exit code:", r.returncode)


## 6 · Export + package the model

No special export tool — the checkpoint **is** a plain `state_dict`, exactly the worker contract. We copy `v1.pth` → `<instrument>48.pth` and pair it with the (identical) training yaml.

In [ ]:
import shutil
pth_src = CKPT_DIR/"v1.pth"
assert pth_src.exists(), f"no checkpoint at {pth_src} — did training run?"
pth_out = OUT_DIR/f"{INSTRUMENT}48.pth"
yml_out = OUT_DIR/f"{INSTRUMENT}48.yaml"
shutil.copy2(pth_src, pth_out)
shutil.copy2(cfg_path, yml_out)

# sanity: the shipped yaml must match training capacity exactly
import torch
sd = torch.load(pth_out, map_location="cpu")
print(f"{pth_out.name}: {pth_out.stat().st_size/1e6:.1f} MB, {len(sd)} tensors")
print(f"{yml_out.name}: paired config")
# zip for easy download from the pod's file browser
shutil.make_archive(str(OUT_DIR/f"{INSTRUMENT}48"), "zip", OUT_DIR, base_dir=None)
print("\n📦 download this from the pod:", OUT_DIR/f"{INSTRUMENT}48.zip")
print("   (contains", pth_out.name, "+", yml_out.name, ")")


## 7 · Drop into the worker (run on your laptop, not the pod)

Download `<instrument>48.zip`, unzip, then in your local **retone** repo:

```bash
cp violin48.pth  runpod/ddsp_models/violin48.pth
cp violin48.yaml runpod/ddsp_models/violin48.yaml
```

Allow the new weights past `.gitignore` (they're `.pth`) — add to `.gitignore`:
```
!runpod/ddsp_models/violin48.pth
```

Edit **`runpod/ddsp_engine.py`** `MODELS` (the *only* code change):
```python
MODELS = {
    "violin":    ("violin48.pth",    "violin48.yaml"),     # 48k replaces the 16k v2
    "saxophone": ("saxophone48.pth", "saxophone48.yaml"),
    "flute":     ("flute48.pth",     "flute48.yaml"),
    "trumpet":   ("trumpet48.pth",   "trumpet48.yaml"),
    "bass":      ("bass48.pth",      "bass48.yaml"),
}
```
Also add each new instrument to the backend's `DDSP_INSTRUMENTS` list in `backend/app/services/tone_transfer.py` so the UI offers it. Then commit + push (HTTP/1.1 to avoid the large-file 408):
```bash
git config http.version HTTP/1.1
git add runpod/ddsp_models/ runpod/ddsp_engine.py backend/app/services/tone_transfer.py .gitignore
git commit -m "v3: 48kHz DDSP models"
git push origin main       # RunPod GitHub build auto-deploys a new endpoint version
```
The worker serves them with **zero** other changes: `render_mono` reads `cfg.sample_rate=48000`, calls `get_f0(...,48000)` + `reconstruction(add_reverb=True)`, and resamples 48 k→44.1 k for the browser.

> **Hard rule:** never hand-edit capacity keys in the shipped yaml — copy the exact training yaml. `strict=False` silently random-inits any shape-mismatched layer (garbage, no error).


## 8 · Next instruments, and where the other families go

**Add another Engine-1 instrument:** change `INSTRUMENT` in §0, refill `raw/`, re-run §2 → §6. Same pipeline, no code changes. Train **violin first**, confirm it beats v2, then batch flute → sax → trumpet → bass on the same rented card (~1 day total).

**Spectrogram note (why Engine 1 already uses them):** DDSP synthesizes in the time domain (oscillators), but it *trains* against a **multi-scale STFT loss** (`MSSLoss([2048,1024,512,256])`) — so these decoders are already spectrogram-optimized. For a sustained tone with a clean f0, an oscillator model beats spectrogram inversion (no vocoder phase artifacts). Where spectrograms/latent-diffusion genuinely win is **polyphony** — which is exactly Engine 2.

**The other two families (not this notebook — pretrained, integration-only):**
- **Engine 2 — piano & guitar → AFTER** (ACIDS-IRCAM control-transfer diffusion; ships a Maestro piano checkpoint + a GuitarSet checkpoint; torch, 44.1 kHz, single-GPU). Audio-driven, polyphonic, spectral/latent-domain — the higher-quality path your spectrogram intuition points to. Integration is inference-only in the worker (resample 48 k↔44.1 k at the boundary). Honest quality: piano ~7/10, guitar ~6/10 (validate guitar hardest — plucks/bends stress it); dense-chord pitch drift is the known risk.
- **Engine 3 — drums → Inverse Drum Machine (IDM)** (Apache-2.0, torch, `idm-44-train-kits` pretrained at 44.1 kHz). Analyze per-instrument onsets + velocities + gains (that *is* the preserved performance), then swap each learned one-shot for a target-kit one-shot and re-sequence. Same groove, new kit. Very good for kick/snare/toms; dense hats/rolls "machine-gun" unless you use velocity-layered/round-robin target one-shots.

Each stem from the 6-stem separation routes to its engine: vocals/bass/mono-lead → **Engine 1**, piano/guitar → **Engine 2**, drums → **Engine 3**.
